# MI ECG PTB-XL Training

Colab notebook for training reproducible myocardial infarction (MI) detection baselines on PTB-XL.

## Runtime

Use **Runtime > Change runtime type > GPU** for deep models. Classical baselines can run on CPU.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/mi_model')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_DIR)

## Bring Project Code Into Colab

Copy this `mi_model` folder to `MyDrive/mi_model`, or clone your GitHub repo and set `PROJECT_DIR` to that clone.

In [ ]:
%cd /content/drive/MyDrive/mi_model
!ls -la

## Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## Quick Smoke Run

Downloads 20 stratified PTB-XL records and verifies the full classical pipeline. Do not treat this as final performance.

In [ ]:
!python scripts/run_classical_experiments.py \
  --max-records 20 \
  --preprocess raw_zscore \
  --models logistic_regression

## Fast Full PTB-XL records100 Download

Run this before full experiments. It uses PhysioNet recursive download, then the experiment scripts can run with `--no-download`.

In [ ]:
%%time
from pathlib import Path
from datetime import datetime

print('Download cell started:', datetime.now().isoformat(timespec='seconds'))
Path('data/ptbxl').mkdir(parents=True, exist_ok=True)

# Metadata files are small. records100 is much larger and contains many WFDB files.
!wget -N -c --show-progress -np -nH --cut-dirs=3 -P data/ptbxl https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv
!wget -N -c --show-progress -np -nH --cut-dirs=3 -P data/ptbxl https://physionet.org/files/ptb-xl/1.0.3/scp_statements.csv

print('Downloading records100. This can take 30-120+ minutes on Google Drive because it has many small files.')
!wget -r -N -c -np -nH --cut-dirs=3 --progress=dot:giga -P data/ptbxl https://physionet.org/files/ptb-xl/1.0.3/records100/

!du -sh data/ptbxl
print('Download cell finished:', datetime.now().isoformat(timespec='seconds'))


## Full Classical Runs

Run both all-12-lead and thesis-inspired wearable 3-differential-lead baselines.


In [ ]:
%%time
!python scripts/run_classical_experiments.py \
  --no-download \
  --lead-mode all12 \
  --preprocess raw_zscore bandpass_0.5_40_zscore bandpass_0.5_40_zscore_downsample50 \
  --models logistic_regression random_forest hist_gradient_boosting

!python scripts/run_classical_experiments.py \
  --no-download \
  --lead-mode wearable3_vdiff \
  --preprocess raw_zscore bandpass_0.5_40_zscore \
  --models logistic_regression random_forest hist_gradient_boosting


## Deep Learning Runs

Recommended on GPU. First train 12-lead raw-waveform models, then train thesis-inspired wearable 3-lead models including an STFT spectrogram CNN.


In [ ]:
%%time
!python scripts/run_deep_experiments.py \
  --no-download \
  --lead-mode all12 \
  --epochs 20 \
  --batch-size 128 \
  --preprocess raw_zscore bandpass_0.5_40_zscore \
  --models resnet1d inception1d

!python scripts/run_deep_experiments.py \
  --no-download \
  --lead-mode wearable3_vdiff \
  --epochs 20 \
  --batch-size 128 \
  --preprocess bandpass_0.5_40_zscore \
  --models resnet1d inception1d spectrogram2d


## Generate Report

In [ ]:
%%time
!python scripts/make_report.py
from pathlib import Path
print(Path('report.md').read_text(encoding='utf-8')[:5000])

## Artifacts

- `reports/metrics_classical.csv`
- `reports/metrics_deep.csv`
- `reports/*.pt` for trained deep models
- `report.md` regenerated from metrics